# Poroelasticity Analysis from Flow Simulations

This notebook processes partial stress data from LAMMPS flow simulations and extracts:
- Volume fractions: $\phi_p(z)$ and $\phi_s(z)$
- Pore pressure: $p_p(z)$
- Network stress: $\boldsymbol{\sigma}'(z)$

**Each trajectory dump is plotted as a separate curve** to show temporal evolution.

## Theory

Partial stress tensors in poroelasticity:

$$\boldsymbol{\sigma}_s = (-\phi_s p_p + A^0_s)\mathbf{I}$$

$$\boldsymbol{\sigma}_p = (-\phi_p p_p + A^0_p)\mathbf{I} + J \boldsymbol{\sigma}'$$

Assuming $A^0_i = 0$ and $J = 1$ (small deformations):

$$p_p(z) = -\frac{\sigma_{s,zz}(z)}{\phi_s(z)}$$

$$\sigma'_{ii}(z) = \sigma_{p,ii}(z) + \phi_p(z) p_p(z)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi
import os
from pathlib import Path

print("Imports successful")

## Configuration

In [ ]:
# ========== USER INPUTS ==========

# Base name for your simulation (without direction or component suffixes)
# Example: if files are "stress_x_polymer_mysim_60000.dat"
# then sim_name = "mysim_60000"
sim_name = "walled_slab_support_tall_angle_8_1.0_1.0_10000000_50000"  # CHANGE THIS

# Paths
stress_data_dir = "../../flow_data_local/partial_stress_data/"
traj_file = f"../../flow_data_local/traj_files/gel_flow_{sim_name}.lammpstrj"
output_folder = "../../flow_data_local/flow_plots/"

# Spatial binning (should match LAMMPS script)
binWidth = 2.0  # Must match LAMMPS variable

# =================================

## Helper Functions

In [ ]:
def read_ave_time_file(filepath):
    """
    Read LAMMPS ave/time output file.
    Format: timestep nrows, then row value pairs.
    
    Returns:
        List of tuples: (timestep, bin_indices, values)
    """
    data_by_time = []
    
    with open(filepath, 'r') as f:
        lines = [line for line in f if not line.startswith('#') and line.strip()]
        
        i = 0
        while i < len(lines):
            parts = lines[i].split()
            if len(parts) == 2:  # Timestep line
                timestep = int(parts[0])
                nrows = int(parts[1])
                values = []
                
                for j in range(1, nrows + 1):
                    if i + j < len(lines):
                        v_parts = lines[i + j].split()
                        if len(v_parts) == 2:
                            values.append(float(v_parts[1]))
                
                if values:
                    bins = np.arange(1, len(values) + 1)
                    data_by_time.append((timestep, bins, np.array(values)))
                
                i += nrows + 1
            else:
                i += 1
    
    return data_by_time


def read_lammpstrj_frame(filepath, frame_idx=0):
    """
    Read a specific frame from LAMMPS trajectory file.
    
    Returns:
        timestep, box_bounds, atoms_data
        atoms_data: list of [id, type, mol, x, y, z]
    """
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Find frame boundaries
    frame_starts = [i for i, line in enumerate(lines) if 'ITEM: TIMESTEP' in line]
    
    if frame_idx >= len(frame_starts) or frame_idx < -len(frame_starts):
        raise ValueError(f"Frame {frame_idx} not found. Only {len(frame_starts)} frames available.")
    
    # Handle negative indexing
    if frame_idx < 0:
        frame_idx = len(frame_starts) + frame_idx
    
    start_idx = frame_starts[frame_idx]
    end_idx = frame_starts[frame_idx + 1] if frame_idx + 1 < len(frame_starts) else len(lines)
    
    frame_lines = lines[start_idx:end_idx]
    
    # Parse timestep
    timestep = int(frame_lines[1].strip())
    
    # Parse number of atoms
    natoms = int(frame_lines[3].strip())
    
    # Parse box bounds
    box_bounds = {}
    box_bounds['x'] = [float(x) for x in frame_lines[5].split()]
    box_bounds['y'] = [float(x) for x in frame_lines[6].split()]
    box_bounds['z'] = [float(x) for x in frame_lines[7].split()]
    
    # Parse atoms (skip header line at index 8)
    atoms_data = []
    for line in frame_lines[9:9+natoms]:
        parts = line.split()
        atom_id = int(parts[0])
        atom_type = int(parts[1])
        mol_id = int(parts[2])
        x, y, z = float(parts[3]), float(parts[4]), float(parts[5])
        atoms_data.append([atom_id, atom_type, mol_id, x, y, z])
    
    return timestep, box_bounds, atoms_data


def compute_volume_fractions_1d(atoms_data, box_bounds, bin_width, direction='z'):
    """
    Compute volume fractions in 1D bins along specified direction.
    
    Uses SIMPLE PARTICLE COUNTING method:
    - Count particles in each bin
    - Divide bin volume equally: phi_p = N_polymer / N_total
    
    This is a SIMPLIFICATION of true Voronoi tessellation:
    - TRUE METHOD: Construct Voronoi cells, sum actual volumes
    - SIMPLIFICATION: Assume each particle occupies equal volume
    
    The simple method misses:
    - Spatial heterogeneity (clustering, voids)
    - Density variations (compressed vs expanded regions)
    - Interface effects (different packing at surfaces)
    
    But it's ~10-100x faster and gives reasonable results for uniform systems.
    
    Args:
        atoms_data: list of [id, type, mol, x, y, z]
        box_bounds: dict with 'x', 'y', 'z' keys containing [lo, hi]
        bin_width: width of bins in specified direction
        direction: 'x', 'y', or 'z'
    
    Returns:
        bin_centers, phi_polymer, phi_solvent
    """
    # Extract positions and types
    atoms = np.array(atoms_data)
    types = atoms[:, 1].astype(int)
    positions = atoms[:, 3:6]
    
    # Atom type definitions:
    # 1,2 = polymer/crosslinker
    # 3 = solvent
    # 4 = support
    # 5 = piston
    # 6 = walls
    
    polymer_mask = (types == 1) | (types == 2)
    solvent_mask = (types == 3)
    
    # Set up binning direction
    if direction == 'z':
        dir_idx = 2
        bin_lo, bin_hi = box_bounds['z']
        perp_dims = (box_bounds['x'][1] - box_bounds['x'][0],
                     box_bounds['y'][1] - box_bounds['y'][0])
    elif direction == 'y':
        dir_idx = 1
        bin_lo, bin_hi = box_bounds['y']
        perp_dims = (box_bounds['x'][1] - box_bounds['x'][0],
                     box_bounds['z'][1] - box_bounds['z'][0])
    else:  # x
        dir_idx = 0
        bin_lo, bin_hi = box_bounds['x']
        perp_dims = (box_bounds['y'][1] - box_bounds['y'][0],
                     box_bounds['z'][1] - box_bounds['z'][0])
    
    # Create bins
    bin_edges = np.arange(bin_lo, bin_hi + bin_width, bin_width)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    nbins = len(bin_centers)
    
    # Bin volume
    bin_volume = bin_width * perp_dims[0] * perp_dims[1]
    
    # Initialize volume arrays
    vol_polymer = np.zeros(nbins)
    vol_solvent = np.zeros(nbins)
    
    # SIMPLE COUNTING METHOD:
    # Each particle in bin gets equal share of bin volume
    for i, (center_lo, center_hi) in enumerate(zip(bin_edges[:-1], bin_edges[1:])):
        # Find atoms in this bin
        in_bin = (positions[:, dir_idx] >= center_lo) & (positions[:, dir_idx] < center_hi)
        
        # Count polymer and solvent
        n_polymer_in_bin = np.sum(polymer_mask & in_bin)
        n_solvent_in_bin = np.sum(solvent_mask & in_bin)
        n_total_in_bin = n_polymer_in_bin + n_solvent_in_bin
        
        if n_total_in_bin > 0:
            # Each particle gets equal share
            # This assumes uniform spacing - the key simplification!
            vol_per_particle = bin_volume / n_total_in_bin
            vol_polymer[i] = n_polymer_in_bin * vol_per_particle
            vol_solvent[i] = n_solvent_in_bin * vol_per_particle
    
    # Calculate volume fractions
    phi_polymer = vol_polymer / bin_volume
    phi_solvent = vol_solvent / bin_volume
    
    return bin_centers, phi_polymer, phi_solvent


print("Helper functions defined")

## Step 0: Read Stress Data

In [ ]:
# Read all 6 stress profiles (x, y, z for polymer and solvent)
stress_files = {
    'polymer_x': f"{stress_data_dir}stress_x_polymer_{sim_name}.dat",
    'polymer_y': f"{stress_data_dir}stress_y_polymer_{sim_name}.dat",
    'polymer_z': f"{stress_data_dir}stress_z_polymer_{sim_name}.dat",
    'solvent_x': f"{stress_data_dir}stress_x_solvent_{sim_name}.dat",
    'solvent_y': f"{stress_data_dir}stress_y_solvent_{sim_name}.dat",
    'solvent_z': f"{stress_data_dir}stress_z_solvent_{sim_name}.dat",
}

stress_data = {}
for key, filepath in stress_files.items():
    print(f"Reading {key}...")
    stress_data[key] = read_ave_time_file(filepath)
    print(f"  Found {len(stress_data[key])} time snapshots")

n_stress_snapshots = len(stress_data['polymer_z'])
print(f"\nTotal stress snapshots: {n_stress_snapshots}")

## Step 1: Process All Timesteps

Loop through all trajectory frames and stress snapshots to calculate:
- Volume fractions (from trajectory)
- Pore pressure (from solvent stress and volume fractions)
- Network stress (from polymer stress and pore pressure)

In [ ]:
print(f"Reading trajectory file: {traj_file}")
print("Counting frames...\n")

# Count frames in trajectory
with open(traj_file, 'r') as f:
    lines = f.readlines()
    frame_starts = [i for i, line in enumerate(lines) if 'ITEM: TIMESTEP' in line]
    n_frames = len(frame_starts)
    print(f"Found {n_frames} trajectory frames")

if n_frames != n_stress_snapshots:
    print(f"WARNING: Trajectory frames ({n_frames}) != stress snapshots ({n_stress_snapshots})")
    print(f"Will process minimum: {min(n_frames, n_stress_snapshots)}")

n_snapshots = min(n_frames, n_stress_snapshots)
print(f"\nProcessing {n_snapshots} snapshots...\n")

# Storage for all snapshots
all_timesteps = []
all_phi_polymer = []
all_phi_solvent = []
all_p_p = []
all_sigma_prime_xx = []
all_sigma_prime_yy = []
all_sigma_prime_zz = []
all_sigma_p = []  # Store as [xx, yy, zz] for each snapshot
all_sigma_s = []  # Store as [xx, yy, zz] for each snapshot

# Process each snapshot
for idx in range(n_snapshots):
    if (idx + 1) % max(1, n_snapshots // 10) == 0 or idx == 0:
        print(f"  Processing snapshot {idx + 1}/{n_snapshots}...")
    
    # Read trajectory frame
    traj_timestep, box_bounds, atoms_data = read_lammpstrj_frame(traj_file, frame_idx=idx)
    
    # Compute volume fractions using simple counting method
    z_centers, phi_polymer, phi_solvent = compute_volume_fractions_1d(
        atoms_data, box_bounds, binWidth, direction='z'
    )
    
    # Get corresponding stress data
    stress_timestep = stress_data['polymer_z'][idx][0]
    bins_z = stress_data['polymer_z'][idx][1]
    z_coords = (bins_z * binWidth - binWidth/2)
    
    sigma_p_xx = stress_data['polymer_x'][idx][2]
    sigma_p_yy = stress_data['polymer_y'][idx][2]
    sigma_p_zz = stress_data['polymer_z'][idx][2]
    
    sigma_s_xx = stress_data['solvent_x'][idx][2]
    sigma_s_yy = stress_data['solvent_y'][idx][2]
    sigma_s_zz = stress_data['solvent_z'][idx][2]
    
    # Calculate pore pressure: p_p = -sigma_s_zz / phi_s
    # Avoid division by zero where there's no solvent
    phi_s_safe = np.where(phi_solvent > 1e-6, phi_solvent, np.nan)
    p_p = -sigma_s_zz / phi_s_safe
    
    # For network stress calculation, fill NaN with 0
    p_p_filled = np.where(np.isnan(p_p), 0.0, p_p)
    
    # Calculate network stresses: sigma'_ii = sigma_p_ii + phi_p * p_p
    sigma_prime_xx = sigma_p_xx + phi_polymer * p_p_filled
    sigma_prime_yy = sigma_p_yy + phi_polymer * p_p_filled
    sigma_prime_zz = sigma_p_zz + phi_polymer * p_p_filled
    
    # Store results
    all_timesteps.append(stress_timestep)
    all_phi_polymer.append(phi_polymer)
    all_phi_solvent.append(phi_solvent)
    all_p_p.append(p_p)
    all_sigma_prime_xx.append(sigma_prime_xx)
    all_sigma_prime_yy.append(sigma_prime_yy)
    all_sigma_prime_zz.append(sigma_prime_zz)
    all_sigma_p.append([sigma_p_xx, sigma_p_yy, sigma_p_zz])
    all_sigma_s.append([sigma_s_xx, sigma_s_yy, sigma_s_zz])

print(f"\nProcessing complete!")
print(f"Timesteps: {all_timesteps[0]} to {all_timesteps[-1]}")
print(f"Total snapshots analyzed: {len(all_timesteps)}")

# Store z-coordinates and box bounds from last frame for plotting
z_lo, z_hi = box_bounds['z']
Lz = z_hi - z_lo
z_norm = (z_coords - z_lo) / Lz
z_centers_norm = (z_centers - z_lo) / Lz

## Step 2: Generate Plots

Create 4 subplots with each timestep as a separate curve.
Color progression shows temporal evolution (early = purple, late = yellow).

In [ ]:
# Create output directory
os.makedirs(output_folder, exist_ok=True)

# Set up color scheme for temporal progression
colors = plt.cm.viridis(np.linspace(0, 1, n_snapshots))

# Create figure with 4 subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Poroelasticity Analysis: {sim_name}\n{n_snapshots} snapshots from t={all_timesteps[0]} to t={all_timesteps[-1]}', 
             fontsize=12, fontweight='bold')

# ========== Plot (a): Volume Fractions ==========
ax = axes[0, 0]
for idx in range(n_snapshots):
    alpha = 0.3 if n_snapshots > 5 else 0.7
    label_p = r'$\phi_p$' if idx == n_snapshots - 1 else None
    label_s = r'$\phi_s$' if idx == n_snapshots - 1 else None
    
    ax.plot(z_centers_norm, all_phi_polymer[idx], '-', 
            color=colors[idx], linewidth=1.5, alpha=alpha, label=label_p)
    ax.plot(z_centers_norm, all_phi_solvent[idx], '--', 
            color=colors[idx], linewidth=1.5, alpha=alpha, label=label_s)

ax.set_xlabel('z / Lz', fontsize=11)
ax.set_ylabel('Volume Fraction', fontsize=11)
ax.set_title('(a) Volume Fractions', fontweight='bold')
if n_snapshots <= 5:
    ax.legend(loc='best', fontsize=9)
else:
    # Add manual legend for many curves
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], color='gray', lw=2, label=r'$\phi_p$ (solid)'),
                      Line2D([0], [0], color='gray', lw=2, ls='--', label=r'$\phi_s$ (dashed)')]
    ax.legend(handles=legend_elements, loc='best', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)

# ========== Plot (b): Pore Pressure ==========
ax = axes[0, 1]
for idx in range(n_snapshots):
    alpha = 0.3 if n_snapshots > 5 else 0.7
    label = f't={all_timesteps[idx]}' if idx == 0 or idx == n_snapshots - 1 else None
    ax.plot(z_norm, all_p_p[idx], '-', 
            color=colors[idx], linewidth=1.5, alpha=alpha, label=label)

ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('z / Lz', fontsize=11)
ax.set_ylabel('Pore Pressure', fontsize=11)
ax.set_title('(b) Pore Pressure Profile', fontweight='bold')
if n_snapshots <= 10:
    ax.legend(loc='best', fontsize=8)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)

# ========== Plot (c): Partial Stress Tensors ==========
ax = axes[1, 0]
for idx in range(n_snapshots):
    alpha = 0.3 if n_snapshots > 5 else 0.7
    
    # Polymer stresses (blue shades, solid/dashed/dotted)
    label_pxx = r'$\sigma_{p,xx}$' if idx == n_snapshots - 1 else None
    label_pyy = r'$\sigma_{p,yy}$' if idx == n_snapshots - 1 else None
    label_pzz = r'$\sigma_{p,zz}$' if idx == n_snapshots - 1 else None
    
    ax.plot(z_norm, all_sigma_p[idx][0], '-', 
            color=colors[idx], linewidth=1, alpha=alpha, label=label_pxx)
    ax.plot(z_norm, all_sigma_p[idx][1], '--', 
            color=colors[idx], linewidth=1, alpha=alpha, label=label_pyy)
    ax.plot(z_norm, all_sigma_p[idx][2], ':', 
            color=colors[idx], linewidth=1.5, alpha=alpha, label=label_pzz)
    
    # Solvent stresses (red shades, same line styles)
    label_sxx = r'$\sigma_{s,xx}$' if idx == n_snapshots - 1 else None
    label_syy = r'$\sigma_{s,yy}$' if idx == n_snapshots - 1 else None
    label_szz = r'$\sigma_{s,zz}$' if idx == n_snapshots - 1 else None
    
    # Use different color for solvent - shift in colormap
    s_color = plt.cm.plasma(np.linspace(0, 1, n_snapshots))[idx]
    ax.plot(z_norm, all_sigma_s[idx][0], '-', 
            color=s_color, linewidth=1, alpha=alpha, label=label_sxx)
    ax.plot(z_norm, all_sigma_s[idx][1], '--', 
            color=s_color, linewidth=1, alpha=alpha, label=label_syy)
    ax.plot(z_norm, all_sigma_s[idx][2], ':', 
            color=s_color, linewidth=1.5, alpha=alpha, label=label_szz)

ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('z / Lz', fontsize=11)
ax.set_ylabel('Partial Stress', fontsize=11)
ax.set_title('(c) Partial Stress Tensors', fontweight='bold')
if n_snapshots <= 3:
    ax.legend(loc='best', fontsize=7, ncol=2)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)

# ========== Plot (d): Network Stress ==========
ax = axes[1, 1]
for idx in range(n_snapshots):
    alpha = 0.3 if n_snapshots > 5 else 0.7
    label_xx = r"$\sigma'_{xx}$" if idx == n_snapshots - 1 else None
    label_yy = r"$\sigma'_{yy}$" if idx == n_snapshots - 1 else None
    label_zz = r"$\sigma'_{zz}$" if idx == n_snapshots - 1 else None
    
    ax.plot(z_norm, all_sigma_prime_xx[idx], '-', 
            color=colors[idx], linewidth=1, alpha=alpha, label=label_xx)
    ax.plot(z_norm, all_sigma_prime_yy[idx], '--', 
            color=colors[idx], linewidth=1, alpha=alpha, label=label_yy)
    ax.plot(z_norm, all_sigma_prime_zz[idx], ':', 
            color=colors[idx], linewidth=1.5, alpha=alpha, label=label_zz)

ax.axhline(y=0, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('z / Lz', fontsize=11)
ax.set_ylabel('Network Stress', fontsize=11)
ax.set_title('(d) Network Stress', fontweight='bold')
if n_snapshots <= 5:
    ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
ax.set_xlim(0, 1)

# Add colorbar to show time progression
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import Normalize
cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
norm = Normalize(vmin=all_timesteps[0], vmax=all_timesteps[-1])
cb = ColorbarBase(cax, cmap='viridis', norm=norm, orientation='vertical')
cb.set_label('Timestep', fontsize=10)

plt.tight_layout(rect=[0, 0, 0.9, 1])

# Save figure
output_path = Path(output_folder) / f'poroelasticity_{sim_name}.png'
plt.savefig(output_path, dpi=150, bbox_inches='tight')
print(f"\nPlot saved to: {output_path}")

plt.show()

## Optional: Save Processed Data

In [ ]:
# Save all processed data to file for later analysis
output_data_path = Path(output_folder) / f'poroelasticity_data_{sim_name}.npz'

# Convert lists to arrays for saving
np.savez(output_data_path,
         z_coords=z_coords,
         z_norm=z_norm,
         z_centers=z_centers,
         z_centers_norm=z_centers_norm,
         timesteps=np.array(all_timesteps),
         phi_polymer=np.array(all_phi_polymer),
         phi_solvent=np.array(all_phi_solvent),
         p_p=np.array(all_p_p),
         sigma_prime_xx=np.array(all_sigma_prime_xx),
         sigma_prime_yy=np.array(all_sigma_prime_yy),
         sigma_prime_zz=np.array(all_sigma_prime_zz),
         sigma_p=np.array(all_sigma_p),
         sigma_s=np.array(all_sigma_s),
         sim_name=sim_name,
         n_snapshots=n_snapshots)

print(f"Data saved to: {output_data_path}")
print(f"Data shape: {n_snapshots} timesteps × {len(z_coords)} spatial bins")

## Summary Statistics

In [ ]:
print("\n" + "="*70)
print("SUMMARY STATISTICS (Last Timestep)")
print("="*70)
print(f"\nSimulation: {sim_name}")
print(f"Timesteps analyzed: {all_timesteps[0]} to {all_timesteps[-1]} ({n_snapshots} snapshots)")

# Statistics from last snapshot
phi_p_last = all_phi_polymer[-1]
phi_s_last = all_phi_solvent[-1]
p_p_last = all_p_p[-1]
sigma_prime_xx_last = all_sigma_prime_xx[-1]
sigma_prime_yy_last = all_sigma_prime_yy[-1]
sigma_prime_zz_last = all_sigma_prime_zz[-1]

print(f"\nVolume Fractions:")
print(f"  <φ_polymer> = {phi_p_last.mean():.4f} ± {phi_p_last.std():.4f}")
print(f"  <φ_solvent> = {phi_s_last.mean():.4f} ± {phi_s_last.std():.4f}")
print(f"  <φ_total>   = {(phi_p_last + phi_s_last).mean():.4f} ± {(phi_p_last + phi_s_last).std():.4f}")

valid_mask = ~np.isnan(p_p_last)
if np.sum(valid_mask) > 0:
    print(f"\nPore Pressure (where solvent present):")
    print(f"  <p_p> = {np.nanmean(p_p_last):.4f} ± {np.nanstd(p_p_last):.4f}")
    print(f"  Range: [{np.nanmin(p_p_last):.4f}, {np.nanmax(p_p_last):.4f}]")

print(f"\nNetwork Stress:")
print(f"  <σ'_xx> = {sigma_prime_xx_last.mean():.4f} ± {sigma_prime_xx_last.std():.4f}")
print(f"  <σ'_yy> = {sigma_prime_yy_last.mean():.4f} ± {sigma_prime_yy_last.std():.4f}")
print(f"  <σ'_zz> = {sigma_prime_zz_last.mean():.4f} ± {sigma_prime_zz_last.std():.4f}")

print(f"\nPartial Stress - Polymer:")
sigma_p_last = all_sigma_p[-1]
print(f"  <σ_p,xx> = {sigma_p_last[0].mean():.4f} ± {sigma_p_last[0].std():.4f}")
print(f"  <σ_p,yy> = {sigma_p_last[1].mean():.4f} ± {sigma_p_last[1].std():.4f}")
print(f"  <σ_p,zz> = {sigma_p_last[2].mean():.4f} ± {sigma_p_last[2].std():.4f}")

print(f"\nPartial Stress - Solvent:")
sigma_s_last = all_sigma_s[-1]
print(f"  <σ_s,xx> = {sigma_s_last[0].mean():.4f} ± {sigma_s_last[0].std():.4f}")
print(f"  <σ_s,yy> = {sigma_s_last[1].mean():.4f} ± {sigma_s_last[1].std():.4f}")
print(f"  <σ_s,zz> = {sigma_s_last[2].mean():.4f} ± {sigma_s_last[2].std():.4f}")

print("\n" + "="*70)
print(f"Analysis complete! Processed {n_snapshots} snapshots.")
print("="*70)